# BTC Perpetual Futures: Wavelet Jump Classification & Probabilistic Price Forecasting

A comprehensive research pipeline for detecting price jumps, classifying them using wavelet scattering, and predicting outcomes with uncertainty calibration.

## 🚀 System Overview
1. **Jump Detection**: Statistical Gumbel-threshold method with intraday seasonality.
2. **Feature Engineering**: 63-feature matrix (29 Wavelet, CVD, OI, Volume Profile).
3. **Regimes**: Macro HMM (4-states) and Micro Efficiency regimes.
4. **ML Model Stack**: Dual LightGBM stack for Jump Type -> Outcome.
5. **Uncertainty**: GP-based gating for signal filtering.

## 1. Setup & Imports

In [5]:
!pip install hmmlearn lightgbm scipy

  Using cached hmmlearn-0.3.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 4.1 MB/s eta 0:00:00


In [16]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pywt
import lightgbm as lgb
import scipy.stats as stats
from typing import List, Dict, Tuple
from tqdm.notebook import tqdm
from hmmlearn.hmm import GaussianHMM
from scipy.stats import linregress
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.ensemble import IsolationForest
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

# Settings
class CONFIG:
    SYMBOL = "BTCUSDT"
    # NEW — replace with these 4 lines:
    TRAIN_START = "2021-01-01"
    TRAIN_END   = "2022-12-31"
    VAL_START   = "2023-01-01"
    VAL_END     = "2023-06-30"

    JUMP_SIGMA_WINDOW = 120
    JUMP_THRESHOLD = 4.0
    JUMP_CLUSTER_GAP = 10

    WAVELET_J = 6
    WAVELET_NAME = 'cmor1.5-1.0'

    MACRO_HMM_STATES = 4

    LABEL_ENDO_VOL_BUILDUP = 1.5
    LABEL_ENDO_SCAT_ASYM   = 0.0
    LABEL_ANTI_VOL_BUILDUP = 0.8
    LABEL_ANTI_TREND_ALIGN = 0.55

    BUCKET_STRONG_UP   =  0.493
    BUCKET_MILD_UP     =  0.112
    BUCKET_FLAT_LO     = -0.131
    BUCKET_MILD_DOWN   = -0.512

np.random.seed(42)
print("Environment initialized.")

Environment initialized.


## 2. Data loading
We load klines from `dataset/Market Data` and funding from `dataset/BTCUSDT-fundingRate-2026-03`.

In [17]:
def load_data():
    klines_path = "/content/Market data"
    funding_path = "/content/Funding Rate"

    k_files = sorted([f for f in os.listdir(klines_path) if f.endswith(".csv")])
    k_list = []
    for f in tqdm(k_files, desc="Klines"):
        tmp = pd.read_csv(os.path.join(klines_path, f), header=0, names=[
            "open_time", "open", "high", "low", "close", "volume",
            "close_time", "quote_vol", "n_trades", "taker_buy_base", "taker_buy_quote", "ignore"
        ])
        tmp["open_time"] = pd.to_datetime(tmp["open_time"], unit="ms", utc=True)
        k_list.append(tmp.set_index("open_time"))

    df = pd.concat(k_list).sort_index()
    df["delta"] = df["taker_buy_base"] - (df["volume"] - df["taker_buy_base"])
    df["cvd"] = df["delta"].cumsum()

    f_list = []
    for root, _, files in os.walk(funding_path):
        for f in files:
            if f.endswith(".csv"):
                tmp = pd.read_csv(os.path.join(root, f))
                tmp["calc_time"] = pd.to_datetime(tmp["calc_time"], unit="ms", utc=True)
                f_list.append(tmp.rename(columns={"last_funding_rate": "funding_rate"}).set_index("calc_time")[["funding_rate"]])

    df_f = pd.concat(f_list).sort_index() if f_list else pd.DataFrame(columns=["funding_rate"])
    return df.astype(float), df_f

df_klines, df_funding = load_data()
print(f"Loaded {len(df_klines)} bars and {len(df_funding)} funding entries.")

Klines:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded 2103840 bars and 4383 funding entries.


## 3. Jump Detection Model

In [23]:
def detect_jumps(df):
    rets = np.log(df["close"] / df["close"].shift(1))
    sigma = rets.rolling(CONFIG.JUMP_SIGMA_WINDOW).std()
    f = rets.abs().groupby(rets.index.hour).transform('mean') / (rets.abs().mean() + 1e-12)

    score = rets / (f * sigma + 1e-12)
    is_jump = score.abs() > CONFIG.JUMP_THRESHOLD

    jumps = df[is_jump].copy()
    jumps["direction"] = np.sign(score[is_jump])

    # Cluster filter
    filtered = []
    last = None
    for ts, row in jumps.iterrows():
        if last is None or (ts - last).total_seconds() / 60 > CONFIG.JUMP_CLUSTER_GAP:
            filtered.append(row)
            last = ts
    return pd.DataFrame(filtered)

df_jumps = detect_jumps(df_klines)
print(f"Detected {len(df_jumps)} price jumps.")

Detected 8989 price jumps.


## 4. Market Regime Detection (Macro & Micro)

In [24]:
class RegimeDetector:
    def __init__(self):
        self.hmm = GaussianHMM(n_components=CONFIG.MACRO_HMM_STATES, covariance_type="diag", n_iter=100)

    def fit_macro(self, df, funding_df):
        # Resample to daily
        daily = df.resample('1D').last()
        daily['vol'] = np.log(daily['close'] / daily['close'].shift(1)).rolling(20).std()
        # Handle missing funding
        f_avg = funding_df.resample('1D').mean()
        daily['funding'] = f_avg['funding_rate'] if 'funding_rate' in f_avg.columns else 0.0

        data = daily[['vol', 'funding']].dropna()
        self.hmm.fit(data.values)
        return pd.Series(self.hmm.predict(data.values), index=data.index)

    def detect_micro(self, df_slice, cvd_slope, cvd_div):
        # Efficiency Ratio
        net_move = abs(df_slice["close"].iloc[-1] - df_slice["close"].iloc[-15])
        total_path = df_slice["close"].diff().abs().iloc[-14:].sum()
        er = net_move / (total_path + 1e-12)

        if er > 0.35 and abs(cvd_slope) > 0.01: return 1 # Trending
        if er > 0.35 and cvd_div < 0.3: return 2 # Exhaustion
        return 0 # Balanced

regime_detector = RegimeDetector()
daily_regimes = regime_detector.fit_macro(df_klines, df_funding)
print("Macro HMM fitted.")

Macro HMM fitted.


## 5. Feature Extraction & Matrix Assembly

In [25]:
class FeatureExtractor:
    # NEW — replace the entire extract_wavelet method body with this:
    def extract_wavelet(self, window_rets):
        local_std = np.std(window_rets) + 1e-12
        x = window_rets / local_std

        feats = {}

        # First-order: complex CWT at t=0 (last point of pre-jump window)
        # cmor1.5-1.0 produces complex coefficients.
        # Real part = amplitude. Imaginary part = asymmetry signal.
        for j in range(1, CONFIG.WAVELET_J + 1):
            scale = 2 ** j
            coeffs, _ = pywt.cwt(x, [scale], CONFIG.WAVELET_NAME)
            feats[f'wavelet_real_j{j}'] = float(np.real(coeffs[0, -1]))
            feats[f'wavelet_imag_j{j}'] = float(np.imag(coeffs[0, -1]))

        # Second-order scattering: CWT of |CWT| — captures volatility asymmetry
        # scat_imag_j{j1}_{j2} is genuinely imaginary, not a product of real numbers
        vol_profile = np.abs(x)
        for j1 in range(1, CONFIG.WAVELET_J + 1):
            for j2 in range(j1 + 1, CONFIG.WAVELET_J + 1):
                mod_c, _ = pywt.cwt(vol_profile, [2**j1], CONFIG.WAVELET_NAME)
                scat_c, _ = pywt.cwt(np.abs(mod_c[0]), [2**j2], CONFIG.WAVELET_NAME)
                feats[f'scat_imag_j{j1}_{j2}'] = float(np.imag(scat_c[0, -1]))

        feats['vol_buildup_ratio'] = float(
            np.std(x[15:]) / (np.std(x[:15]) + 1e-12)
        )
        feats['trend_alignment'] = float(np.mean(x > 0))
        return feats

    def extract_micro(self, df_slice, funding, funding_8h):
        cvd = df_slice["cvd"].iloc[-15:]
        price = df_slice["close"].iloc[-15:]
        s = linregress(range(len(cvd)), cvd.values).slope if len(cvd)>1 else 0
        d = float(np.corrcoef(price.values, cvd.values)[0, 1]) if len(cvd)>1 else 0

        feats = {
            "cvd_slope": s,
            "cvd_price_div": d,
            "delta_ratio": float(df_slice["delta"].iloc[-1] / (df_slice["volume"].iloc[-1] + 1e-12)),
            "funding": float(funding),
            "funding_change": float(funding - funding_8h),
            "micro_regime": float(regime_detector.detect_micro(df_slice, s, d))
        }
        return feats

extractor = FeatureExtractor()
feature_rows = []
for ts, row in tqdm(df_jumps.iterrows(), total=len(df_jumps)):
    idx = df_klines.index.get_loc(ts)
    if idx < 30: continue

    pre_window = df_klines.iloc[idx-30 : idx]
    jump_slice = df_klines.iloc[idx-30 : idx+1]

    w_rets = np.log(pre_window["close"] / pre_window["close"].shift(1)).fillna(0).values * row["direction"]
    w_f = extractor.extract_wavelet(w_rets)

    f_val = df_funding.asof(ts)["funding_rate"] if not df_funding.empty else 0.0
    f_8h = df_funding.asof(ts - pd.Timedelta(hours=8))["funding_rate"] if not df_funding.empty else 0.0
    m_f = extractor.extract_micro(jump_slice, f_val, f_8h)

    # Macro Regime
    m_r = daily_regimes.asof(ts)
    m_f["macro_regime"] = float(m_r) if pd.notna(m_r) else 0.0

    ret_1h = np.nan
    if idx + 60 < len(df_klines):
        ret_1h = np.log(df_klines["close"].iloc[idx+60] / df_klines["close"].iloc[idx]) * 100

    feature_rows.append({**w_f, **m_f, "target_ret": ret_1h, "timestamp": ts})

fm = pd.DataFrame(feature_rows).set_index("timestamp").dropna()
print(f"Feature matrix build: {fm.shape}")

  0%|          | 0/8989 [00:00<?, ?it/s]

Feature matrix build: (8987, 37)


In [26]:
# Diagnose actual scat_imag distribution
scat_cols = [c for c in fm.columns if 'scat_imag' in c]
scat_means = fm[scat_cols].mean(axis=1)
print(f"scat_asym distribution:")
print(f"  min:    {scat_means.min():.6f}")
print(f"  max:    {scat_means.max():.6f}")
print(f"  mean:   {scat_means.mean():.6f}")
print(f"  median: {scat_means.median():.6f}")
print(f"  pct below 0.0: {(scat_means < 0.0).mean():.1%}")
print(f"  pct below -0.001: {(scat_means < -0.001).mean():.1%}")
print(f"vol_buildup_ratio > 1.5: {(fm['vol_buildup_ratio'] > 1.5).mean():.1%}")
print(f"Both conditions met: {((fm['vol_buildup_ratio'] > 1.5) & (scat_means < 0.0)).mean():.1%}")

scat_asym distribution:
  min:    0.000000
  max:    0.332760
  mean:   0.159487
  median: 0.157766
  pct below 0.0: 0.0%
  pct below -0.001: 0.0%
vol_buildup_ratio > 1.5: 25.2%
Both conditions met: 0.0%


## 6. Labeling & Modeling

In [27]:
# Replace assign_wavelet_label with a scoring approach:
def assign_wavelet_label(row):
    scat_cols = [c for c in row.index if 'scat_imag' in c]
    scat_asym = np.mean([row[c] for c in scat_cols])

    # Score-based: both signals contribute, neither is a hard gate
    vol_score  = (row['vol_buildup_ratio'] - 1.0)   # positive = buildup
    scat_score = -scat_asym                           # negative asym = endogenous

    endo_score = vol_score * 0.6 + scat_score * 0.4

    if endo_score > 0.3:
        return 'endogenous'
    if row['vol_buildup_ratio'] < 0.8 and row['trend_alignment'] > 0.55:
        return 'anticipatory'
    return 'exogenous'

fm["jump_type"] = fm.apply(assign_wavelet_label, axis=1)
# OLD — nothing here, goes straight to outcome bucketing

# NEW — add these lines immediately after the jump_type assignment:
# Verify label distribution before training — catches degenerate labelling immediately
label_dist = fm["jump_type"].value_counts()
print(f"Jump type distribution:\n{label_dist}\n")
dominant_pct = label_dist.max() / label_dist.sum()
if len(label_dist) < 2 or dominant_pct > 0.95:
    raise ValueError(
        f"DEGENERATE LABELS: {label_dist.idxmax()} = {dominant_pct:.1%} of all jumps.\n"
        "Fix wavelet implementation or labeller thresholds before proceeding."
    )
print("Label distribution OK.\n")

fm["outcome"] = fm["target_ret"].apply(lambda r: "strong_up" if r>=CONFIG.BUCKET_STRONG_UP else ("mild_up" if r>=CONFIG.BUCKET_MILD_UP else ("flat" if r>=CONFIG.BUCKET_FLAT_LO else ("mild_down" if r>=CONFIG.BUCKET_MILD_DOWN else "strong_down"))))

train_df = fm[fm.index <= CONFIG.TRAIN_END]
val_df = fm[(fm.index > CONFIG.TRAIN_END) & (fm.index <= CONFIG.VAL_END)]

X_cols = [c for c in fm.columns if c not in ["target_ret", "jump_type", "outcome"]]
le_o = LabelEncoder()
y_tr, y_vl = le_o.fit_transform(train_df["outcome"]), le_o.transform(val_df["outcome"])

clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.03, verbose=-1, class_weight='balanced')
clf.fit(train_df[X_cols], y_tr)

# Full corrected training block — replace from clf = ... to probs = ...:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split

# Split training data: 80% for LightGBM, 20% for calibration fitting
X_tr_raw  = train_df[X_cols].values
y_tr_raw  = y_tr
split_idx = int(len(X_tr_raw) * 0.8)

X_lgbm, X_calib = X_tr_raw[:split_idx], X_tr_raw[split_idx:]
y_lgbm, y_calib = y_tr_raw[:split_idx], y_tr_raw[split_idx:]

clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.03,
                          verbose=-1, class_weight='balanced')
clf.fit(X_lgbm, y_lgbm)

calibrated_clf = CalibratedClassifierCV(clf, method='isotonic', cv='prefit')
calibrated_clf.fit(X_calib, y_calib)

probs = calibrated_clf.predict_proba(val_df[X_cols].values)
probs = calibrated_clf.predict_proba(val_df[X_cols])

#probs = clf.predict_proba(val_df[X_cols])
mean_brier = np.mean([brier_score_loss((y_vl == i).astype(int), probs[:, i]) for i in range(5)])
print(f"Validation Mean Brier Score: {mean_brier:.4f}")


# OLD — cell ends here

# NEW — add full per-class calibration breakdown:
print("\n── PER-CLASS CALIBRATION ──")
class_names = le_o.classes_
for i, name in enumerate(class_names):
    y_bin = (y_vl == i).astype(int)
    b = brier_score_loss(y_bin, probs[:, i])
    try:
        frac_pos, mean_pred = calibration_curve(y_bin, probs[:, i], n_bins=8)
        ece = float(np.mean(np.abs(frac_pos - mean_pred)))
    except ValueError:
        ece = float("nan")
    status = "PASS" if b < 0.20 else "FAIL"
    print(f"  {name:<15}: Brier={b:.4f}  ECE={ece:.4f}  {status}")

# Confidence-stratified check
print("\n── CONFIDENCE DEVIATION CHECK ──")
confidence = probs.max(axis=1)
predicted  = probs.argmax(axis=1)
correct    = (predicted == y_vl)
for lo, hi in [(0.4, 0.5), (0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.0)]:
    mask = (confidence >= lo) & (confidence < hi)
    if mask.sum() < 20:
        continue
    actual_acc = correct[mask].mean()
    pred_conf  = confidence[mask].mean()
    deviation  = abs(actual_acc - pred_conf)
    edge = "EDGE" if deviation < 0.05 else "NO EDGE"
    print(f"  Conf {lo:.0%}-{hi:.0%}: n={mask.sum():4d}  "
          f"predicted={pred_conf:.1%}  actual={actual_acc:.1%}  "
          f"dev={deviation:.1%}  {edge}")

# Jump type accuracy report
print("\n── JUMP TYPE DISTRIBUTION ──")
print(fm["jump_type"].value_counts().to_string())
print(f"\nTrain: {len(train_df)} jumps | Val: {len(val_df)} jumps")

Jump type distribution:
jump_type
exogenous       6605
endogenous      1832
anticipatory     550
Name: count, dtype: int64

Label distribution OK.



/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Validation Mean Brier Score: 0.1527

── PER-CLASS CALIBRATION ──
  flat           : Brier=0.2200  ECE=0.0715  FAIL
  mild_down      : Brier=0.1656  ECE=0.1979  PASS
  mild_up        : Brier=0.1961  ECE=0.1362  PASS
  strong_down    : Brier=0.0786  ECE=0.3241  PASS
  strong_up      : Brier=0.1034  ECE=0.1244  PASS

── CONFIDENCE DEVIATION CHECK ──
  Conf 40%-50%: n= 270  predicted=43.5%  actual=40.7%  dev=2.7%  EDGE
  Conf 50%-60%: n= 114  predicted=53.1%  actual=36.0%  dev=17.2%  NO EDGE

── JUMP TYPE DISTRIBUTION ──
jump_type
exogenous       6605
endogenous      1832
anticipatory     550

Train: 2547 jumps | Val: 1266 jumps


## 7. Uncertainty Gating & Visuals

In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.pipeline import Pipeline

# The GP with RBF fails in high dimensions because all points appear
# equidistant from each other (curse of dimensionality).
# Solution: reduce to 10 principal components before fitting.
# PCA preserves the main variance structure while making the
# feature space tractable for the RBF kernel.
gpc_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=10, random_state=42)),
    ('gpc',    GaussianProcessClassifier(
        kernel=1.0 * RBF(
            length_scale=1.0,
            length_scale_bounds=(1e-2, 1e4)   # tighter bounds after PCA normalises scale
        ) + WhiteKernel(
            noise_level=0.1,
            noise_level_bounds=(1e-3, 1e2)    # tighter lower bound prevents near-zero noise
        ),
        random_state=42,
        n_restarts_optimizer=5,
    ))
])

n_sub   = min(800, len(train_df))
sub_idx = np.random.RandomState(42).choice(len(train_df), n_sub, replace=False)
gpc_pipeline.fit(train_df[X_cols].values[sub_idx], y_tr[sub_idx])

gp_probs   = gpc_pipeline.predict_proba(val_df[X_cols].values)
gated_mask = gp_probs.max(axis=1) < 0.40   # lower threshold — 5 classes so 0.4 is meaningful
print(f"GP Gating: {gated_mask.mean():.1%} of signals filtered")
print(f"GP kept signals: {(~gated_mask).sum()} out of {len(val_df)}")

print("\n── GP CONFIDENCE DISTRIBUTION ──")
gp_conf = gp_probs.max(axis=1)
for lo, hi in [(0.0,0.4),(0.4,0.5),(0.5,0.6),(0.6,0.7),(0.7,1.0)]:
    n   = ((gp_conf >= lo) & (gp_conf < hi)).sum()
    pct = n / len(gp_conf) * 100
    print(f"  {lo:.0%}-{hi:.0%}: {n:4d} signals ({pct:.1f}%)")

# Verify GP is not degenerate
if gp_probs.max(axis=1).mean() < 0.30:
    print("\nWARNING: GP still outputting near-uniform probabilities.")
    print("Increase PCA components or adjust kernel bounds.")
else:
    print(f"\nGP mean confidence: {gp_probs.max(axis=1).mean():.1%} — working correctly")

/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceW

GP Gating: 100.0% of signals filtered
GP kept signals: 0 out of 1266

── GP CONFIDENCE DISTRIBUTION ──
  0%-40%: 1266 signals (100.0%)
  40%-50%:    0 signals (0.0%)
  50%-60%:    0 signals (0.0%)
  60%-70%:    0 signals (0.0%)
  70%-100%:    0 signals (0.0%)

Increase PCA components or adjust kernel bounds.


In [29]:
# Replace the entire cell 15 with this:
from sklearn.ensemble import RandomForestClassifier

rf_uncertainty = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

n_sub   = min(800, len(train_df))
sub_idx = np.random.RandomState(42).choice(len(train_df), n_sub, replace=False)
rf_uncertainty.fit(train_df[X_cols].values[sub_idx], y_tr[sub_idx])

rf_probs   = rf_uncertainty.predict_proba(val_df[X_cols].values)
gated_mask = rf_probs.max(axis=1) < 0.35
print(f"RF Uncertainty Gating: {gated_mask.mean():.1%} of signals filtered")
print(f"RF kept signals: {(~gated_mask).sum()} out of {len(val_df)}")

print("\n── RF CONFIDENCE DISTRIBUTION ──")
rf_conf = rf_probs.max(axis=1)
for lo, hi in [(0.0,0.3),(0.3,0.4),(0.4,0.5),(0.5,0.6),(0.6,0.7),(0.7,1.0)]:
    n   = ((rf_conf >= lo) & (rf_conf < hi)).sum()
    pct = n / len(rf_conf) * 100
    print(f"  {lo:.0%}-{hi:.0%}: {n:4d} signals ({pct:.1f}%)")

mean_conf = rf_conf.mean()
if mean_conf < 0.30:
    print(f"\nWARNING: RF mean confidence {mean_conf:.1%} is near random. Check features.")
else:
    print(f"\nRF mean confidence: {mean_conf:.1%}")

RF Uncertainty Gating: 100.0% of signals filtered
RF kept signals: 0 out of 1266

── RF CONFIDENCE DISTRIBUTION ──
  0%-30%: 1263 signals (99.8%)
  30%-40%:    3 signals (0.2%)
  40%-50%:    0 signals (0.0%)
  50%-60%:    0 signals (0.0%)
  60%-70%:    0 signals (0.0%)
  70%-100%:    0 signals (0.0%)

